# PATH MANAGEMENT

In [1]:
import os

print(os.getcwd())
if not os.getcwd().endswith("app"):
    os.chdir("../app")
    print(os.getcwd())

import pandas as pd
pd.set_option('display.max_rows', 500)
pd.set_option('display.max_columns', 500)

%load_ext autoreload
%autoreload 2
# %matplotlib inline

/home/turbotowerlnx/Documents/Master/TA/TA-Spanish-Esperanto-Translator/notebooks
/home/turbotowerlnx/Documents/Master/TA/TA-Spanish-Esperanto-Translator/app


In [2]:
from src.config import Configuration

CONFIG = Configuration(
    max_tok_length=16,
    batch_size=64,
)

# Baseline

In this notebook, we are going to learn how to use Meta's large pre-trained model [NLLB](https://huggingface.co/docs/transformers/model_doc/nllb) on the [Europarl-ST dataset](https://huggingface.co/datasets/tj-solergibert/Europarl-ST), but only that [dataset of Europarl-ST focused on the text data for MT from English](https://huggingface.co/datasets/tj-solergibert/Europarl-ST-processed-mt-en).

Europarl-ST is already available in the [Datasets repository](https://huggingface.co/datasets) from Hugging Face. The [Datasets library](https://huggingface.co/docs/datasets) makes easy to access and load datasets. For example, you can easily load your own dataset following [this tutorial](https://huggingface.co/docs/datasets/loading#local-and-remote-files).

In [3]:
# from datasets import load_dataset

# raw_datasets = load_dataset("tj-solergibert/Europarl-ST-processed-mt-en")

# print(raw_datasets)

from src.data import get_es_eo_dataset

raw_datasets = get_es_eo_dataset(CONFIG)

print(raw_datasets)

/home/turbotowerlnx/Documents/Master/TA/TA-Spanish-Esperanto-Translator/venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


DatasetDict({
    train: Dataset({
        features: ['source_text', 'dest_text', 'dest_lang'],
        num_rows: 200965
    })
    test: Dataset({
        features: ['source_text', 'dest_text', 'dest_lang'],
        num_rows: 43063
    })
    valid: Dataset({
        features: ['source_text', 'dest_text', 'dest_lang'],
        num_rows: 43063
    })
})


As shown, the Europarl-ST already comes with a pre-defined partition on the three conventional sets: training, validation and test. Each set is a dictionary with a list of source sentences (source_text), target sentences (dest_text) and the target language (dest_lang).

Let's take a closer look at the features of the training set:

In [4]:
raw_datasets["train"].features

{'source_text': Value('string'),
 'dest_text': Value('string'),
 'dest_lang': Value('int64')}

As you can see, the possible target languages are German, English, Spanish, French, Italian, Dutch, Polish, Portuguese and Romanian.

Let us take a look at the translations of the first two English sentences:

In [5]:
raw_datasets["train"][:14]["source_text"]

['artículo anteriorse devela un misterio: ¿por qué moni argento era de tostado?',
 'en el siglo iii surgieron un número de tribus germánicas del oeste grandes: alemanni, francos, catos, \ufeffsajones, frisii, \ufeffsicambri, y thuringii\ufeff.',
 'hubo unos 200 invitados.',
 '¿eres tú mayor de edad?',
 '-"pero no tienes dinero, ¿verdad?"',
 'así que, cuando ella te deja, ¿de donde crees que ella va a hacer a continuación.',
 'para empezar, como ya hemos dicho, debemos tomar la fruta con el estómago vacío.',
 'soy una viuda con cuatro hijos y me quedé atrapado en una situación financiera desde abril de 2016 y necesitaba refinanciar y pagar mis cuentas.',
 'el trabajo con espacios en blanco debe comenzar a fines de la primavera o principios del verano y no retrasarse hasta el otoño para evitar problemas e interrupciones.',
 'es la riqueza guardada por su dueño para su propia desgracia.',
 'buscamos una canción que trate sobre alguno de los siguientes temas: «desarrollo global» o «un solo

In [6]:

raw_datasets["train"][:14]["dest_text"]

['estis mistero por la polico : kial ŝteli nur unu ŝuon anstataù paro ?',
 'la tria jarcento vidis la aperon de kelkaj grandaj okcident ĝermanaj triboj: la alemanoj, frankoj, bavarii-, ĥatoj, saksoj, frisii, sicambri, kaj thuringii.',
 'venis ĉirkaŭ 200 gastoj.',
 'ĉu vi estas la plej aĝa?',
 '"sed vi ne posedas tiom da mono, ĉu ne?"',
 'do, kiam ŝi lasas vin, kie vi kredas, ke ŝi faros poste.',
 'kiel antaŭe menciite, la drogo devas esti prenita sur malplena stomako.',
 'en ĉi tiu tempo mi estas vidvino kun kvar infanoj kaj mi estis ligita en financa situacio en majo 2018 kaj bezonis refinanci kaj pagi miajn biletojn.',
 'laboro kun spacoj devas komenciĝi fine de printempo aŭ frua somero kaj ne malhelpu ĝis aŭtuno por eviti problemojn kaj interrompojn.',
 'riĉecon konservatan por la malutilo de ĝia propra mastro.',
 'tie ĉi mi menciu nur unu temaron, tiun de tutmondiĝo aŭ „globaliĝo”.',
 'ni serĉu rekte la titolon «orientaj tapiŝoj»!',
 'tamen, ĉu tranĉeoj estas por ke ni koncentriĝu'

In [7]:
raw_datasets["train"][:14]["dest_lang"]

[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]

As shown, each English sentence is repeated for each of the seven target languages (0: 'de', 2: 'es', 3: 'fr', 4: 'it', 5: 'nl', 6: 'pl', 7: 'pt').

Provided that the NLLB model was pretrained on sentence pairs involving 200 languages, being one of the them the translation from English into Spanish, we are going to be filtering Europarl-ST only for English into Spanish using a simple [lambda function](https://realpython.com/python-lambda/) with the [Dataset.filter() function](https://huggingface.co/docs/datasets/v2.9.0/en/package_reference/main_classes#datasets.Dataset.filter).

In [8]:
# lang="es"
# lang_id = raw_datasets["train"].features["dest_lang"].names.index(lang)
# raw_datasets = raw_datasets.filter(lambda x: x["dest_lang"] == lang_id)

Now we load the pre-trained tokenizer for the NLLB model and apply it to the English-Spanish pair:

In [9]:
from transformers import AutoTokenizer

checkpoint = "facebook/nllb-200-distilled-600M"
# from flores200_codes import flores_codes
# src_code = "eng_Latn"
# tgt_code = "spa_Latn"
tokenizer = AutoTokenizer.from_pretrained(
    checkpoint, 
    padding=True, 
    pad_to_multiple_of=8, 
    src_lang=CONFIG.src_code, 
    tgt_lang=CONFIG.tgt_code, 
    truncation=True, 
    max_length=CONFIG.max_tok_length,
    )

We can apply the tokenizer function to any dataset taking advantage that Hugging Face Datasets are [Apache Arrow](https://arrow.apache.org) files stored on the disk, so you only keep the samples you ask for loaded in memory.

To keep the data as a dataset, we will use the [Dataset.map() function](https://huggingface.co/docs/datasets/en/package_reference/main_classes#datasets.Dataset.map). This also allows us some extra flexibility, if we need more preprocessing done than just tokenization. The map() method works by applying a function on each element of the dataset.

In our case, each sample pair is going to be preprocessed according to the training needs of the model that is to be used:

In [10]:
def preprocess_function(sample):
    model_inputs = tokenizer(
        sample["source_text"], 
        text_target = sample["dest_text"],
        )
    return model_inputs


The way the Datasets library applies this processing is by adding new fields to the datasets, one for each key in the dictionary returned by the tokenize function, that is, *input_ids*, *attention_mask* and *labels*. We can check what the preprocess_function is doing with a small sample

In [11]:
sample = raw_datasets["train"].select(range(2))
model_input = preprocess_function({
    "source_text": list(sample["source_text"]),
    "dest_text": list(sample["dest_text"]),
})
print(model_input)

{'input_ids': [[256161, 37679, 42599, 201, 79, 9342, 159, 214320, 248144, 2247, 2298, 12250, 22550, 8394, 1384, 2250, 79, 81617, 1080, 248130, 2], [256161, 153, 336, 49192, 77975, 2562, 443, 9435, 159, 31897, 79, 3690, 13052, 3045, 24158, 128615, 597, 127227, 34753, 248144, 34883, 20144, 248079, 203773, 248079, 7875, 57, 248079, 127, 2646, 33, 248079, 1000, 39076, 248079, 22347, 711, 174, 248079, 35, 185350, 1694, 248062, 81, 2]], 'attention_mask': [[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1], [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]], 'labels': [[256048, 14366, 1551, 42803, 956, 82, 2741, 629, 18816, 126860, 12329, 34585, 7382, 7742, 12329, 216544, 223836, 248742, 25100, 385, 2], [256048, 82, 143179, 12207, 248080, 1384, 70658, 82, 538, 9435, 79, 153950, 4710, 10701, 911, 248080, 5509, 13662, 14, 8249, 248086, 3690, 27768, 248144, 82, 34883, 46619, 248079, 11071, 25086, 2

In [12]:
for sample in model_input['input_ids']:
    print(tokenizer.convert_ids_to_tokens(sample))

['spa_Latn', '▁artículo', '▁anterior', 'se', '▁de', 'vela', '▁un', '▁misterio', ':', '▁¿', 'por', '▁qué', '▁moni', '▁arg', 'ento', '▁era', '▁de', '▁tost', 'ado', '?', '</s>']
['spa_Latn', '▁en', '▁el', '▁siglo', '▁iii', '▁sur', 'gi', 'eron', '▁un', '▁número', '▁de', '▁tri', 'bus', '▁ger', 'má', 'nicas', '▁del', '▁oeste', '▁grandes', ':', '▁alem', 'anni', ',', '▁francos', ',', '▁cat', 'os', ',', '▁sa', 'jon', 'es', ',', '▁fr', 'isii', ',', '▁sic', 'amb', 'ri', ',', '▁y', '▁thur', 'ingi', 'i', '▁.', '</s>']


We can recover the source text by applying [batch_decode](https://huggingface.co/docs/transformers/en/internal/tokenization_utils#transformers.PreTrainedTokenizerBase.batch_decode) of the tokenizer 

In [13]:
tokenizer.batch_decode(model_input['input_ids'])

['spa_Latn artículo anteriorse devela un misterio: ¿por qué moni argento era de tostado?</s>',
 'spa_Latn en el siglo iii surgieron un número de tribus germánicas del oeste grandes: alemanni, francos, catos, sajones, frisii, sicambri, y thuringii .</s>']

Now, we can apply the preprocess_function to the raw datasets (training, validation and test):

In [14]:
tokenized_datasets = raw_datasets.map(preprocess_function, batched=True)

Map: 100%|██████████| 43063/43063 [00:01<00:00, 38046.44 examples/s]


We are going to filter the tokenized datasets by maximum number of tokens in source and target language:

In [15]:
tokenized_datasets = tokenized_datasets.filter(lambda x: len(x["input_ids"]) <= CONFIG.max_tok_length and len(x["labels"]) <= CONFIG.max_tok_length , desc=f"Discarding source and target sentences with more than {CONFIG.max_tok_length} tokens")

Discarding source and target sentences with more than 16 tokens: 100%|██████████| 200965/200965 [00:02<00:00, 70769.00 examples/s]
Discarding source and target sentences with more than 16 tokens: 100%|██████████| 43063/43063 [00:00<00:00, 74449.86 examples/s]
Discarding source and target sentences with more than 16 tokens: 100%|██████████| 43063/43063 [00:00<00:00, 60735.75 examples/s]


We can take a quick look at the length histogram in the source language:

In [16]:
dic = {}
for sample in tokenized_datasets['train']:
    sample_length = len(sample['input_ids'])
    if sample_length not in dic:
        dic[sample_length] = 1
    else:
        dic[sample_length] += 1 

for i in range(1,CONFIG.max_tok_length+1):
    if i in dic:
        print(f"{i:>2} {dic[i]:>3}")

 4 105
 5 927
 6 3388
 7 7242
 8 11147
 9 13183
10 13548
11 12196
12 10276
13 8202
14 5968
15 4257
16 2772


Checking a sample after filtering by maximum number of tokens:

In [17]:
for sample in tokenized_datasets['train'].select(range(5)):
    print(sample['input_ids'])
    print(sample['attention_mask'])
    print(sample['labels'])

[256161, 164994, 42462, 884, 33536, 2776, 248075, 2]
[1, 1, 1, 1, 1, 1, 1, 1]
[256048, 71692, 129913, 884, 7287, 9416, 248075, 2]
[256161, 2247, 5134, 4020, 20115, 79, 49401, 248130, 2]
[1, 1, 1, 1, 1, 1, 1, 1, 1]
[256048, 51297, 382, 4114, 82, 42995, 9, 81938, 248130, 2]
[256161, 55530, 42668, 254, 49643, 55881, 248079, 2247, 204608, 1278, 2]
[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]
[256048, 69, 34468, 382, 167, 1123, 69016, 70576, 170, 8991, 248079, 51297, 167, 1278, 2]
[256161, 388, 51783, 4979, 9788, 7217, 57, 132818, 1115, 4868, 8423, 68031, 2]
[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]
[256048, 159914, 248079, 51297, 9667, 121447, 3031, 4114, 956, 264, 158, 35270, 174, 108723, 2]
[256161, 219, 254, 42164, 437, 340, 5766, 124385, 248079, 130, 40085, 79, 55881, 248075, 2]
[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]
[256048, 130, 9714, 167, 37883, 1495, 43172, 83448, 248079, 83448, 1495, 8991, 248075, 2]


bitsandbytes is a quantization library with a Transformers integration. With this integration, you can quantize a model to 8 or 4-bits and enable many other options by configuring the BitsAndBytesConfig class. For example, you can:

<ul>
<li>set load_in_4bit=True to quantize the model to 4-bits when you load it</li>
<li>set bnb_4bit_quant_type="nf4" to use a special 4-bit data type for weights initialized from a normal distribution</li>
<li>set bnb_4bit_use_double_quant=True to use a nested quantization scheme to quantize the already quantized weights</li>
<li>set bnb_4bit_compute_dtype=torch.bfloat16 to use bfloat16 for faster computation</li>
</ul>


In [18]:
import torch
from transformers import BitsAndBytesConfig

quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.bfloat16,
)

Pass the quantization_config to the from_pretrained method.

In [19]:
from transformers import AutoModelForSeq2SeqLM

model = AutoModelForSeq2SeqLM.from_pretrained(
    checkpoint,
    quantization_config=quantization_config
    )


## Evaluation

The last thing to define for our Seq2SeqTrainer is how to compute the metrics to evaluate the predictions of our model with respect to references. To this purpose, we use the [Evaluate library](https://huggingface.co/docs/evaluate) which includes the definition of generic and task-specific metrics. In our case, we use the [BLEU metric](https://huggingface.co/spaces/evaluate-metric/bleu), or to be more precise, [sacreBLEU](https://huggingface.co/spaces/evaluate-metric/sacrebleu). You can see a simple example of usage below:

:

In [20]:
from evaluate import load

metric_bleu = load("sacrebleu")
metric_comet = load("comet")

/home/turbotowerlnx/Documents/Master/TA/TA-Spanish-Esperanto-Translator/venv/lib/python3.12/site-packages/torchmetrics/utilities/imports.py:23: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import DistributionNotFound, get_distribution
Fetching 5 files: 100%|██████████| 5/5 [00:00<00:00, 117817.53it/s]
Lightning automatically upgraded your loaded checkpoint from v1.8.3.post1 to v2.5.6. To apply the upgrade to your files permanently, run `python -m pytorch_lightning.utilities.upgrade_checkpoint ../../../../../.cache/huggingface/hub/models--Unbabel--wmt22-comet-da/snapshots/2760a223ac957f30acfb18c8aa649b01cf1d75f2/checkpoints/model.ckpt`
Encoder model frozen.
/home/turbotowerlnx/Documents/Master/TA/TA-Spanish-Esperanto-Translator/venv/lib/python3.12/site-packages/pytorch_lig

We need to define a function compute_metrics to compute BLEU scores at each epoch. The example below performs a basic post-processing to decode the predictions into texts:

In [ ]:
import numpy as np
def postprocess_text(preds, labels):
    preds = [pred.strip() for pred in preds]
    labels = [[label.strip()] for label in labels]

    return preds, labels

def compute_metrics(preds, labels, sources):
    # Convert to lists if coming from a datasets.Column
    if not isinstance(labels, list):
        labels = list(labels)
        
    if isinstance(preds, tuple):
        preds = preds[0]
    decoded_preds = tokenizer.batch_decode(preds, skip_special_tokens=True)

    # Replace negative ids in the labels as we can't decode them.
    labels = [
        [tokenizer.pad_token_id if j < 0 else j for j in label]
        for label in labels
    ]
    decoded_labels = tokenizer.batch_decode(labels, skip_special_tokens=True)
    
    # Decode sources
    # decoded_sources = tokenizer.batch_decode(sources, skip_special_tokens=True)

    # Some simple post-processing
    decoded_preds, decoded_labels = postprocess_text(decoded_preds, decoded_labels)

    result_blue = metric_bleu.compute(
        predictions=decoded_preds, 
        references=decoded_labels
    )
    result_comet = metric_comet.compute(
        sources=sources,
        predictions=decoded_preds, 
        references=[label[0] for label in decoded_labels]  # COMET expects flat list, not nested
    )
    result = {
        "bleu": result_blue["score"],
        "comet": result_comet["mean_score"]
    }

    prediction_lens = [np.count_nonzero(pred != tokenizer.pad_token_id) for pred in preds]
    result["gen_len"] = np.mean(prediction_lens)
    result = {k: round(v, 4) for k, v in result.items()}
    return result

## Inference

At inference time, it is recommended to use the [generate function](https://huggingface.co/docs/transformers/main_classes/text_generation). This method takes care of encoding the input and auto-regressively generates the decoder output. Check out [this blog post](https://huggingface.co/blog/how-to-generate) to know all the details about generating text with Transformers.
There’s also [this blog post](https://huggingface.co/blog/encoder-decoder#encoder-decoder) which explains how generation works in general in encoder-decoder models.

Let us first load the default inference parameters of NLLB.

In [22]:
from transformers import GenerationConfig

generation_config = GenerationConfig.from_pretrained(
    checkpoint,
)

print(generation_config)

GenerationConfig {
  "bos_token_id": 0,
  "decoder_start_token_id": 2,
  "eos_token_id": 2,
  "max_length": 200,
  "pad_token_id": 1
}



We prepare the test set in batches to be translated:

In [23]:
batch_tokenized_test = tokenized_datasets['test'].batch(CONFIG.batch_size)

Batching examples: 100%|██████████| 20049/20049 [00:00<00:00, 54767.82 examples/s]


Processing in batches to add padding and convert to tensors, then perform inference with num_beams = 1 and do_sample = False, that is, greedy search.

In [24]:
import tqdm
number_of_batches = len(batch_tokenized_test["source_text"])
all_sources = []
output_sequences = []
for i in tqdm.tqdm(range(number_of_batches)):
    all_sources.extend(batch_tokenized_test["source_text"][i])

    inputs = tokenizer(
        batch_tokenized_test["source_text"][i], 
        max_length=CONFIG.max_tok_length, 
        truncation=True, 
        return_tensors="pt", 
        padding=True,
        )
    with torch.no_grad():    
        output_batch = model.generate(
            generation_config=generation_config, 
            input_ids=inputs["input_ids"].cuda(), 
            attention_mask=inputs["attention_mask"].cuda(), 
            forced_bos_token_id=tokenizer.convert_tokens_to_ids(CONFIG.tgt_code), 
            max_length = CONFIG.max_tok_length, 
            num_beams=1, 
            do_sample=False,
            )
    output_sequences.extend(output_batch.cpu())

100%|██████████| 314/314 [00:48<00:00,  6.42it/s]


In [25]:
result = compute_metrics(output_sequences, tokenized_datasets["test"]["labels"], all_sources)
print(f'BLEU score: {result["bleu"]:0.4f}')
print(f'COMET score: {result["comet"]:0.4f}')

💡 Tip: For seamless cloud uploads and versioning, try installing [litmodels](https://pypi.org/project/litmodels/) to enable LitModelCheckpoint, which syncs automatically with the Lightning model registry.
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
/home/turbotowerlnx/Documents/Master/TA/TA-Spanish-Esperanto-Translator/venv/lib/python3.12/site-packages/torch/__init__.py:1551: UserWarning: Please use the new API settings to control TF32 behavior, such as torch.backends.cudnn.conv.fp32_precision = 'tf32' or torch.backends.cuda.matmul.fp32_precision = 'ieee'. Old settings, e.g, torch.backends.cuda.matmul.allow_tf32 = True, torch.backends.cudnn.allow_tf32 = True, allowTF32CuDNN() and allowTF32CuBLAS() will be deprecated after Pytorch 2.9. Please see https://pytorch.org/docs/main/notes/cuda.html#tensorfloat-32-tf32-on-ampere-and-later-devices (Triggered internally at /pytorch/aten/src/ATen/Context.cpp:80.)
  return _C._get_float32_matmul_precision()
You a

BLEU score: 19.1038
COMET score: 0.8006


In [27]:
from maikol_utils.print_utils import print_separator

# all_sources already contains raw text strings
# output_sequences contains token IDs that need to be decoded
decoded_outputs = tokenizer.batch_decode(output_sequences, skip_special_tokens=True)

# Get reference translations from test set
test_references = tokenized_datasets["test"]["dest_text"]

# Print first 10 examples
for i, (source, output, reference) in enumerate(zip(all_sources[:10], decoded_outputs[:10], test_references[:10])):
    print_separator(f"Example {i+1}:")
    print(f"Source:      {source}")
    print(f"Translation: {output}")
    print(f"Reference:   {reference}")

________________________________________________________________
                           Example 1:                           

Source:      abro los ojos, deslumbrado.
Translation: Mi malfermas la okulojn, blinde.
Reference:   mi malfermas okulojn, kapturnadas.
________________________________________________________________
                           Example 2:                           

Source:      ellos preferirán la cremación.
Translation: Ili preferas la kremacion.
Reference:   ttt-ejoj vi preferas krei.
________________________________________________________________
                           Example 3:                           

Source:      el servicio de restaurante es sólo para grupos.
Translation: La restoracia servo estas nur por grupoj.
Reference:   hospitality club estas nur por membroj.
________________________________________________________________
                           Example 4:                           

Source:      garantizo sin embargo que será emoc